In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 0. 데이터 불러오기: 2-5 train_test_comparison에서 제거된 이상치까지 반영된 데이터셋
# ============================================================

train_full = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '0_train_full_clean.parquet')
test = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '0_test_future.parquet')

print(train_full.shape, test.shape)

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

(749609, 34) (114111, 34)


In [2]:
# ============================================================
# 0. 라이브러리 불러오기
# ============================================================

import numpy as np

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error

In [3]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_t',
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]


# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_rain_1h',
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_rain_1h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols
num_cols_fe_wl = num_cols + [
    col for col in wl_diff_cols if col != 'wl_t'
]
num_cols_fe = num_cols_fe_wl + [
    col for col in rain_int_cols if col not in ['imerg_rain_1h', 'sfc_rain_1h']
]

print(f"""<변수개수 검증>
    수위변수: {len(wl_cols)}
    강우변수: {len(rain_cols)}, 
    환경변수: {len(env_cols)}, 
    수위차분(wl_t포함): {len(wl_diff_cols)},
    강우강도(rain_1h포함): {len(rain_int_cols)}
    기존 숫자열(수위 + 강우 + 환경): {len(num_cols)}, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): {len(num_cols_fe_wl)},
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): {len(num_cols_fe)}
""")

<변수개수 검증>
    수위변수: 6
    강우변수: 9, 
    환경변수: 7, 
    수위차분(wl_t포함): 6,
    강우강도(rain_1h포함): 6
    기존 숫자열(수위 + 강우 + 환경): 22, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): 27,
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): 31



In [4]:
# ============================================================
# 1. Tidal / Normal 데이터 분리
# ============================================================

# 조석 영향 의심 관측소 9개 (기존과 동일)
tidal_stations = [
    'ST0110', 'ST0326', 'ST0159',
    'ST0358', 'ST0347', 'ST0415',
    'ST0292', 'ST0313', 'ST0234'
]

# Train 분리
tidal_train = train_full.loc[
    train_full['station_id'].isin(tidal_stations)
].copy()

normal_train = train_full.loc[
    ~train_full['station_id'].isin(tidal_stations)
].copy()

# Test 분리
tidal_test = test.loc[
    test['station_id'].isin(tidal_stations)
].copy()

normal_test = test.loc[
    ~test['station_id'].isin(tidal_stations)
].copy()

# 분리 확인
assert len(tidal_train) + len(normal_train) == len(train_full)
assert len(tidal_test) + len(normal_test) == len(test)

print(f'Train: {len(train_full):,}')
print(f'  Tidal: {len(tidal_train):,}')
print(f'  Normal: {len(normal_train):,}')

print(f'\nTest: {len(test):,}')
print(f'  Tidal: {len(tidal_test):,}')
print(f'  Normal: {len(normal_test):,}')

Train: 749,609
  Tidal: 14,251
  Normal: 735,358

Test: 114,111
  Tidal: 2,169
  Normal: 111,942


In [5]:
# ============================================================
# 2. 독립변수 / 종속변수 분리
# ============================================================

# Tidal
X_tidal_train = tidal_train[num_cols_fe + ['station_id']].copy()
X_tidal_test = tidal_test[num_cols_fe + ['station_id']].copy()

y_tidal_train = tidal_train[target_col]
y_tidal_test = tidal_test[target_col]

# Normal
X_normal_train = normal_train[num_cols_fe + ['station_id']].copy()
X_normal_test = normal_test[num_cols_fe + ['station_id']].copy()

y_normal_train = normal_train[target_col]
y_normal_test = normal_test[target_col]

# CatBoost 범주형 변수 처리
for X in [
    X_tidal_train, X_tidal_test,
    X_normal_train, X_normal_test
]:
    X['station_id'] = X['station_id'].astype(str)

In [9]:
# ============================================================
# 3. Tidal 데이터 부스팅모델별 성능 비교
# ============================================================

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.metrics import root_mean_squared_error
import pandas as pd
import gc

preds_tidal = {}
results_tidal = {}

for name in ['CatBoost', 'LightGBM', 'XGBoost']:

    print(f'\n========== {name} 학습 ==========')

    # 1. 모델 정의 및 범주형 변수 처리
    if name == 'CatBoost':
        model = CatBoostRegressor(
            cat_features=['station_id'],
            verbose=False,
            random_state=42
        )

        X_train = X_tidal_train
        X_test = X_tidal_test

    else:
        X_train = X_tidal_train.copy()  # catboost용으로 str로 변환된 상태
        X_test = X_tidal_test.copy()  # 나머지 모델은 'category' 자료형 필요해서

        # station_id 범주형 자료형 변환
        X_train['station_id'] = X_train['station_id'].astype('category')
        X_test['station_id'] = X_test['station_id'].astype('category')

        if name == 'LightGBM':
            model = LGBMRegressor(
                verbosity=-1,
                random_state=42
            )

        elif name == 'XGBoost':
            model = XGBRegressor(
                enable_categorical=True,
                random_state=42
            )

    # 2. 모델별 학습 (기본 설정, Early Stopping 없음)
    if name == 'CatBoost':
        model.fit(
            X_train, y_tidal_train
        )

    elif name == 'LightGBM':
        model.fit(
            X_train, y_tidal_train
        )

    elif name == 'XGBoost':
        model.fit(
            X_train, y_tidal_train,
            verbose=False
        )

    # 3. 예측
    pred_test = model.predict(X_test)
    pred_train = model.predict(X_train)

    # 4. RMSE 계산
    rmse_test = root_mean_squared_error(y_tidal_test, pred_test)
    rmse_train = root_mean_squared_error(y_tidal_train, pred_train)

    # 5. 결과 저장
    preds_tidal[name] = pd.Series(
        pred_test, index=X_tidal_test.index
    )

    results_tidal[name] = {
        'Test RMSE': rmse_test,
        'Train RMSE': rmse_train,
        'Gap': rmse_test - rmse_train
    }

    print(f'\n{name} 결과')
    print('Test RMSE:', rmse_test)
    print('Train RMSE:', rmse_train)
    print('Gap:', rmse_test - rmse_train)

    # 6. 모델 메모리 정리
    del model
    gc.collect()

# 7. 모델별 성능 비교표
result_df = pd.DataFrame.from_dict(
    results_tidal,
    orient='index'
)

display(result_df.sort_values('Test RMSE'))


========== CatBoost 학습 ==========

CatBoost 결과
Test RMSE: 0.35925757219053134
Train RMSE: 0.23576076658529677
Gap: 0.12349680560523457

========== LightGBM 학습 ==========

LightGBM 결과
Test RMSE: 0.37600386558495336
Train RMSE: 0.27886601288677654
Gap: 0.09713785269817682

========== XGBoost 학습 ==========

XGBoost 결과
Test RMSE: 0.38655563612291394
Train RMSE: 0.17079418878929903
Gap: 0.2157614473336149


,Test RMSE,Train RMSE,Gap
CatBoost,0.359258,0.235761,0.123497
LightGBM,0.376004,0.278866,0.097138
XGBoost,0.386556,0.170794,0.215761


In [10]:

# ============================================================
# 4. 배깅모델을 위한 Tidal 데이터 OneHotEncoding
# ============================================================

from sklearn.preprocessing import OneHotEncoder

# 1. OneHotEncoder 정의 및 학습
ohe = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

ohe.fit(X_tidal_train[['station_id']])

# 2. Train / Test 원핫인코딩
X_tidal_train_ohe = pd.concat([
    X_tidal_train[num_cols_fe],
    pd.DataFrame(
        ohe.transform(X_tidal_train[['station_id']]),
        columns=ohe.get_feature_names_out(),
        index=X_tidal_train.index
    )
], axis=1)

X_tidal_test_ohe = pd.concat([
    X_tidal_test[num_cols_fe],
    pd.DataFrame(
        ohe.transform(X_tidal_test[['station_id']]),
        columns=ohe.get_feature_names_out(),
        index=X_tidal_test.index
    )
], axis=1)

print(X_tidal_train.shape, X_tidal_train_ohe.shape)
print(X_tidal_test.shape, X_tidal_test_ohe.shape)

(14251, 32) (14251, 40)
(2169, 32) (2169, 40)


In [11]:
# ============================================================
# 5. Tidal RF / ExtraTrees 모델별 성능 비교
# ============================================================

from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import root_mean_squared_error
import pandas as pd
import gc

# 앞에서 학습한 부스팅모델 결과 유지
# preds_tidal, results_tidal

for name in ['RandomForest', 'ExtraTrees']:

    print(f'\n========== {name} 학습 ==========')

    # 1. 모델 정의
    if name == 'RandomForest':
        model = RandomForestRegressor(
            random_state=42,
            n_jobs=-1,
            verbose=0
        )

    elif name == 'ExtraTrees':
        model = ExtraTreesRegressor(
            random_state=42,
            n_jobs=-1,
            verbose=0
        )

    # 2. 모델 학습
    model.fit(X_tidal_train_ohe, y_tidal_train)

    # 3. 예측
    pred_test = model.predict(X_tidal_test_ohe)
    pred_train = model.predict(X_tidal_train_ohe)

    # 4. RMSE 계산
    rmse_test = root_mean_squared_error(y_tidal_test, pred_test)
    rmse_train = root_mean_squared_error(y_tidal_train, pred_train)

    # 5. 결과 저장
    preds_tidal[name] = pd.Series(
        pred_test, index=X_tidal_test.index
    )

    results_tidal[name] = {
        'Test RMSE': rmse_test,
        'Train RMSE': rmse_train,
        'Gap': rmse_test - rmse_train
    }

    print(f'\n{name} 결과')
    print('Test RMSE:', rmse_test)
    print('Train RMSE:', rmse_train)
    print('Gap:', rmse_test - rmse_train)

    # 6. 메모리 정리
    del model
    gc.collect()

# 7. 전체 5개 모델 성능 비교
result_df = pd.DataFrame.from_dict(
    results_tidal,
    orient='index'
)

display(result_df.sort_values('Test RMSE'))


========== RandomForest 학습 ==========

RandomForest 결과
Test RMSE: 0.36311874864856736
Train RMSE: 0.12809241036842658
Gap: 0.23502633828014077

========== ExtraTrees 학습 ==========

ExtraTrees 결과
Test RMSE: 0.3459668004683099
Train RMSE: 2.8884645963809996e-15
Gap: 0.345966800468307


,Test RMSE,Train RMSE,Gap
ExtraTrees,0.345967,2.888465e-15,0.345967
CatBoost,0.359258,2.357608e-01,0.123497
RandomForest,0.363119,1.280924e-01,0.235026
LightGBM,0.376004,2.788660e-01,0.097138
XGBoost,0.386556,1.707942e-01,0.215761


In [12]:
# ============================================================
# 6. 성능이 우수한 CatBoost + ExtraTrees 앙상블 성능 비교
# ============================================================

# 1. 단순 평균 앙상블 (각각 50%)
pred_ensemble = (
    preds_tidal['CatBoost'] * 0.5
    + preds_tidal['ExtraTrees'] * 0.5
)

# 2. RMSE 계산
rmse_ensemble = root_mean_squared_error(
    y_tidal_test,
    pred_ensemble
)

# 3. 기존 결과에 추가
results_tidal['Ensemble_50_50'] = {
    'Test RMSE': rmse_ensemble,
    'Train RMSE': None,
    'Gap': None
}

# 4. 성능 비교
result_df = pd.DataFrame.from_dict(
    results_tidal,
    orient='index'
)

display(result_df.sort_values('Test RMSE'))

,Test RMSE,Train RMSE,Gap
Ensemble_50_50,0.342282,NaN,NaN
ExtraTrees,0.345967,2.888465e-15,0.345967
CatBoost,0.359258,2.357608e-01,0.123497
RandomForest,0.363119,1.280924e-01,0.235026
LightGBM,0.376004,2.788660e-01,0.097138
XGBoost,0.386556,1.707942e-01,0.215761


In [13]:
# ============================================================
# 7. CatBoost + ExtraTrees 앙상블 가중치 최적화
# ============================================================

ensemble_results = []

for w_cat in np.arange(0, 1.001, 0.05):

    pred_ensemble = (
        preds_tidal['CatBoost'] * w_cat
        + preds_tidal['ExtraTrees'] * (1 - w_cat)
    )

    rmse = root_mean_squared_error(
        y_tidal_test,
        pred_ensemble
    )

    ensemble_results.append({
        'CatBoost_weight': round(w_cat, 2),
        'ExtraTrees_weight': round(1 - w_cat, 2),
        'Test RMSE': rmse
    })

ensemble_df = pd.DataFrame(ensemble_results)

display(
    ensemble_df.sort_values('Test RMSE').head(10)
)

,CatBoost_weight,ExtraTrees_weight,Test RMSE
7,0.35,0.65,0.341176
6,0.30,0.70,0.341230
8,0.40,0.60,0.341334
5,0.25,0.75,0.341496
9,0.45,0.55,0.341703
4,0.20,0.80,0.341973
10,0.50,0.50,0.342282
3,0.15,0.85,0.342660
11,0.55,0.45,0.343071
2,0.10,0.90,0.343555


In [14]:
# ============================================================
# 8. 최종 모델 학습 및 Test 통합 RMSE 평가
#    Tidal : CatBoost 45% + ExtraTrees 55%
#    Normal: CatBoost 100%
# ============================================================

from catboost import CatBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import root_mean_squared_error
import pandas as pd
import gc

# ============================================================
# 1. Tidal CatBoost 기준모델
# ============================================================

cat_tidal = CatBoostRegressor(
    iterations=3000,
    random_state=42,
    cat_features=['station_id'],
    verbose=200
)

cat_tidal.fit(
    X_tidal_train, y_tidal_train,
    eval_set=(X_tidal_test, y_tidal_test),
    early_stopping_rounds=100
)

pred_cat_tidal = cat_tidal.predict(X_tidal_test)


# ============================================================
# 2. Tidal ExtraTrees 기본모델
# ============================================================

et_tidal = ExtraTreesRegressor(
    random_state=42,
    n_jobs=-1,
    verbose=0
)

et_tidal.fit(
    X_tidal_train_ohe,
    y_tidal_train
)

pred_et_tidal = et_tidal.predict(X_tidal_test_ohe)


# ============================================================
# 3. Tidal 앙상블 (CatBoost 45% + ExtraTrees 55%)
# ============================================================

pred_tidal = pd.Series(
    pred_cat_tidal * 0.45 + pred_et_tidal * 0.55,
    index=X_tidal_test.index
)

rmse_tidal = root_mean_squared_error(
    y_tidal_test, pred_tidal
)

print(f'Tidal RMSE: {rmse_tidal:.6f}')


# ============================================================
# 4. Normal CatBoost 기준모델
# ============================================================

cat_normal = CatBoostRegressor(
    iterations=3000,
    random_state=42,
    cat_features=['station_id'],
    verbose=200
)

cat_normal.fit(
    X_normal_train, y_normal_train,
    eval_set=(X_normal_test, y_normal_test),
    early_stopping_rounds=100
)

pred_normal = pd.Series(
    cat_normal.predict(X_normal_test),
    index=X_normal_test.index
)

rmse_normal = root_mean_squared_error(
    y_normal_test, pred_normal
)

print(f'Normal RMSE: {rmse_normal:.6f}')


# ============================================================
# 5. Tidal + Normal 예측값 통합
# ============================================================

pred_test_final = pd.concat([
    pred_tidal,
    pred_normal
]).reindex(test.index)

# 통합 데이터 검증
assert len(pred_test_final) == len(test)
assert pred_test_final.notna().all()
assert pred_test_final.index.equals(test.index)


# ============================================================
# 6. 최종 통합 RMSE
# ============================================================

rmse_total = root_mean_squared_error(
    test[target_col],
    pred_test_final
)

print('\n========== 최종 성능 ==========')
print(f'Tidal RMSE : {rmse_tidal:.6f}')
print(f'Normal RMSE: {rmse_normal:.6f}')
print(f'Total RMSE : {rmse_total:.6f}')


# ============================================================
# 7. 메모리 정리
# ============================================================

del cat_tidal, et_tidal, cat_normal
gc.collect()

Learning rate set to 0.039537
0:	learn: 1.7085461	test: 1.7659914	best: 1.7659914 (0)	total: 41.5ms	remaining: 2m 4s
200:	learn: 0.4056550	test: 0.4337543	best: 0.4337543 (200)	total: 8.36s	remaining: 1m 56s
400:	learn: 0.3459636	test: 0.3889485	best: 0.3889485 (400)	total: 16.3s	remaining: 1m 45s
600:	learn: 0.3132075	test: 0.3745648	best: 0.3745648 (600)	total: 24.2s	remaining: 1m 36s
800:	learn: 0.2895007	test: 0.3653392	best: 0.3653392 (800)	total: 32.3s	remaining: 1m 28s
1000:	learn: 0.2719728	test: 0.3620359	best: 0.3619379 (985)	total: 40.6s	remaining: 1m 21s
1200:	learn: 0.2570926	test: 0.3601577	best: 0.3601528 (1193)	total: 49.8s	remaining: 1m 14s
1400:	learn: 0.2445987	test: 0.3590226	best: 0.3589787 (1387)	total: 58.9s	remaining: 1m 7s
1600:	learn: 0.2342170	test: 0.3579406	best: 0.3579406 (1600)	total: 1m 9s	remaining: 1m 1s
1800:	learn: 0.2249139	test: 0.3571290	best: 0.3571289 (1799)	total: 1m 18s	remaining: 52.4s
2000:	learn: 0.2165070	test: 0.3564225	best: 0.3564107 (1

58

In [ ]:
# 모델 구조 선정: 단일 CatBoost와 Tidal/Normal 분리모델의 K-Fold OOF RMSE를 비교한다. 
# Test 결과도 참고하되, K-Fold에서 더 안정적이고 우수한 구조를 우선한다.


# Early Stopping 적용: K-Fold 또는 기존 단일 홀드아웃 Validation에서 확인한 best_iteration_을 활용해 최종 반복 횟수를 결정한다.
# Tidal과 Normal 각각의 최적 반복 횟수를 별도로 확정